Tempo analysis

In [8]:
import pandas as pd

# Ground truth
gt = pd.read_csv("dataset_with_chords_120.csv")

# All detection sheets
xls = pd.ExcelFile("tempo_analysis/tempo_all_songs1.xlsx")
sheets = xls.sheet_names
print("Sheets:", sheets)

detected = {name: pd.read_excel(xls, sheet_name=name) for name in sheets}

print(gt.columns)
for name, df in detected.items():
    print(name, df.columns)

def evaluate_detection(detected_df, ground_truth, tol=0.04):
    df = detected_df.rename(columns={"tempo":"tempo_det"})
    gt = ground_truth.rename(columns={"tempo":"tempo_gt"})
    
    merged = df.merge(gt, on="track_id")
    
    # relative error
    merged["rel_err"] = abs(merged["tempo_det"] - merged["tempo_gt"]) / merged["tempo_gt"]
    merged["correct"] = merged["rel_err"] <= tol
    
    acc = merged["correct"].mean()
    return acc, merged



results = {}
for name, df in detected.items():
    acc, merged = evaluate_detection(df, gt)
    results[name] = acc
    print(f"{name}: accuracy={acc:.2%}")

results_sorted = pd.Series(results).sort_values(ascending=False)
print(results_sorted)


Sheets: ['Bir Damla Gözlerimde_23.2ms,2.9ms', 'Bir Damla Gözlerimde_23.2ms,5.8ms', 'Bir Damla Gözlerimde_46.4ms,5.8ms', 'Bir Damla Gözlerimde_46.4ms,11.6ms', 'Bir Damla Gözlerimde_92.9ms,11.6ms', 'Bir Damla Gözlerimde_92.9ms,23.2ms', 'Bir Damla Gözlerimde_185.8ms,23.2ms', 'Bir Damla Gözlerimde_185.8ms,46.4ms', 'BABYMETAL - KARATE (_23.2ms,2.9ms', 'BABYMETAL - KARATE (_23.2ms,5.8ms', 'BABYMETAL - KARATE (_46.4ms,5.8ms', 'BABYMETAL - KARATE (_46.4ms,11.6ms', 'BABYMETAL - KARATE (_92.9ms,11.6ms', 'BABYMETAL - KARATE (_92.9ms,23.2ms', 'BABYMETAL - KARATE (_185.8ms,23.2ms', 'BABYMETAL - KARATE (_185.8ms,46.4ms', 'Bahara - Audio  I Ha_23.2ms,2.9ms', 'Bahara - Audio  I Ha_23.2ms,5.8ms', 'Bahara - Audio  I Ha_46.4ms,5.8ms', 'Bahara - Audio  I Ha_46.4ms,11.6ms', 'Bahara - Audio  I Ha_92.9ms,11.6ms', 'Bahara - Audio  I Ha_92.9ms,23.2ms', 'Bahara - Audio  I Ha_185.8ms,23.2ms', 'Bahara - Audio  I Ha_185.8ms,46.4ms', 'blink-182 - I Really_23.2ms,2.9ms', 'blink-182 - I Really_23.2ms,5.8ms', 'blink-1

KeyError: 'track_id'

In [11]:
import pandas as pd
import re
from rapidfuzz import process, fuzz

# --- Load data ---
gt = pd.read_csv("dataset_with_chords_120.csv")
xls = pd.ExcelFile("tempo_analysis/tempo_all_songs1.xlsx")

# Function to parse sheet name into (song, window_ms, hop_ms)
def parse_sheet_name(name):
    # Example: "Alec Benjamin - Jesu_46.4ms,5.8ms"
    m = re.match(r"(.+?)_(\d+\.?\d*)ms,(\d+\.?\d*)ms", name)
    if m:
        song = m.group(1).strip()
        window = float(m.group(2))
        hop = float(m.group(3))
        return song, window, hop
    else:
        return name, None, None

# --- Collect detection results ---
rows = []
for sheet in xls.sheet_names:
    song, window, hop = parse_sheet_name(sheet)
    df = pd.read_excel(xls, sheet_name=sheet)
    
    tempo_pred = df["Essentia BPM"].median()
    
    rows.append({
        "track_name_raw": song,
        "window_ms": window,
        "hop_ms": hop,
        "tempo_pred": tempo_pred
    })

detected_summary = pd.DataFrame(rows)

# --- Fuzzy match track_name_raw to gt["track_name"] ---
gt_titles = gt["track_name"].tolist()

def best_match(name, choices, scorer=fuzz.token_sort_ratio, cutoff=70):
    match, score, idx = process.extractOne(name, choices, scorer=scorer)
    if score >= cutoff:
        return match
    return None

detected_summary["track_name"] = detected_summary["track_name_raw"].apply(
    lambda x: best_match(x, gt_titles)
)

# Drop unmatched
detected_matched = detected_summary.dropna(subset=["track_name"])

# --- Merge with ground truth ---
merged = detected_matched.merge(
    gt[["track_name", "tempo"]],
    on="track_name",
    how="inner"
).rename(columns={"tempo":"tempo_gt"})

# --- Evaluate accuracy ---
merged["rel_err"] = abs(merged["tempo_pred"] - merged["tempo_gt"]) / merged["tempo_gt"]
merged["correct"] = merged["rel_err"] <= 0.04   # within 4%

# Accuracy per (window, hop)
accuracy_table = (
    merged.groupby(["window_ms","hop_ms"])["correct"]
    .mean()
    .reset_index()
    .sort_values("correct", ascending=False)
)

# --- Save results ---
merged.to_csv("tempo_eval_detailed.csv", index=False)
accuracy_table.to_csv("tempo_eval_summary.csv", index=False)

print("Saved detailed results → tempo_eval_detailed.csv")
print("Saved summary results → tempo_eval_summary.csv")
print(accuracy_table.head())


Saved detailed results → tempo_eval_detailed.csv
Saved summary results → tempo_eval_summary.csv
   window_ms  hop_ms  correct
0       23.2     2.9      0.0
1       23.2     5.8      0.0
2       46.4     5.8      0.0
3       46.4    11.6      0.0
4       92.9    11.6      0.0
